# 03 — Content Creation

Validate LLM-powered social media post generation:
- Load strategy + insights
- Generate Mastodon posts (max 500 chars)
- Generate Bluesky posts (max 300 chars)
- Test bilingual output (DE + EN)
- Manual quality review

In [ ]:
import os
prefix = os.getenv("S3_STATE_PREFIX", "growth-agent/")
if prefix == "growth-agent/":
    raise RuntimeError(
        f"S3_STATE_PREFIX is {prefix!r} — this is the PRODUCTION prefix. "
        "Set S3_STATE_PREFIX=growth-agent-dev/ in .env before running notebooks."
    )
print(f"Using S3 prefix: {prefix}  ✓")


In [2]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os
import json

load_dotenv('../.env', override=True)

provider = os.environ.get('LLM_PROVIDER', 'ionos')
from agent.llm_client import PROVIDERS
config = PROVIDERS.get(provider)
api_key_set = bool(os.environ.get(config.api_key_env)) if config else False

print(f'Active provider : {provider}')
print(f'API key loaded  : {"yes" if api_key_set else f"NO — set {key_env} in .env"}')

/Users/fredjendrzejewski/GitHub/fretchen.github.io/growth-agent/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Active provider : mistral
API key loaded  : yes


## 1. Load State from Previous Notebooks

In [3]:
from agent.storage import S3Storage
from agent.models import Strategy, Insights

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)

insights_raw = store.read('insights.json')
analysis = store.read('llm_analysis.json')
strategy = Strategy()  # Use defaults

if insights_raw is None or analysis is None:
    print('ERROR: Run notebooks 01 and 02 first.')
else:
    print(f'Insights loaded: {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Growth opportunities: {len(analysis.get("growth_opportunities", []))}')
    print(f'Best pages for social: {len(analysis.get("best_pages_for_social", []))}')

Insights loaded: 43 visitors
Growth opportunities: 5
Best pages for social: 10


## 2. Content Planning — Pick Pages to Promote

In [4]:
from agent.llm_client import LLMClient

llm = LLMClient.from_env()  # provider + API key resolved from LLM_PROVIDER env var

# Build content plan from LLM analysis
pages_to_promote = analysis.get('best_pages_for_social', [])
if not pages_to_promote:
    # Fallback: use top pages from analytics (raw x/y Umami format)
    pages_to_promote = [
        {
            'url': f'https://fretchen.eu{p.get("x", p.get("name", ""))}',
            'title': p.get('x', p.get('name', '')),
            'reason': f'{p["y"]} visitors',
        }
        for p in insights_raw['website_analytics']['top_pages'][:5]
        if p.get('x') or p.get('name')
    ]

print(f'Pages to promote ({len(pages_to_promote)}):')
for p in pages_to_promote:
    print(f'  - {p["url"]} — {p["reason"]}')

Pages to promote (10):
  - /blog/27/ — Strong audience interest
  - /blog/12/ — Relevant topic with existing engagement
  - /quantum/hardware/ — Underrepresented topic worth exploring
  - /amo/13/ — Mystery page with some traffic
  - /de/blog/27/ — Strong audience interest in other languages
  - /blog/ — General blog page with some traffic
  - /de/blog/ — General blog page with some traffic in other languages
  - /amo/10 — Some traffic, but unclear content
  - /amo/11 — Some traffic, but unclear content
  - /blog/12/ — Relevant topic with existing engagement


## 3. Generate Mastodon Post (EN)

In [5]:
# Pick first page to promote
page = pages_to_promote[0]
page_url = page['url']
page_desc = page.get('description', '')

mastodon_prompt = f"""Write a Mastodon post (max 500 characters) about this blog article:

URL: {page_url}?utm_source=mastodon&utm_campaign=growth-agent
Title: {page.get('title', '')}
Article summary: {page_desc}
Why promote: {page['reason']}

Context about the blog: fretchen.eu covers political economy, game theory,
quantum computing, blockchain/Web3, and AI tools.

Requirements:
- Hook in the first line (question or bold claim)
- Mention one specific insight from the article topic
- Include the link
- Add 2-3 relevant hashtags
- Tone: technical but accessible, opinionated
- Language: English

Do NOT use emojis excessively. One is fine.
Return ONLY the post text, nothing else."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'You write engaging social media posts for a technical blog. Be concise and punchy.'},
        {'role': 'user', 'content': mastodon_prompt}
    ],
    temperature=0.8,
    max_tokens=300,
)

mastodon_en = result['content'].strip()
print(f'=== Mastodon Post (EN) [{len(mastodon_en)} chars] ===')
print(mastodon_en)
print(f'\n{"✅ OK" if len(mastodon_en) <= 500 else "❌ TOO LONG"} ({len(mastodon_en)}/500 chars)')

=== Mastodon Post (EN) [387 chars] ===
"Homeownership is the ultimate scam—or is it? A podcast drops truth bombs on the 'American Dream' myth. Spoiler: the math doesn’t add up for most. Renting isn’t throwing money away; it’s buying flexibility in an uncertain economy.

Dive into the cold, hard numbers: https://fretchen.eu/blog/27/?utm_source=mastodon&utm_campaign=growth-agent

#HousingCrisis #EconTwitter #PersonalFinance"

✅ OK (387/500 chars)


## 4. Generate Bluesky Post (EN)

In [6]:
bluesky_prompt = f"""Write a Bluesky post (max 300 characters) about this blog article:

URL: {page_url}?utm_source=bluesky&utm_campaign=growth-agent
Title: {page.get('title', '')}
Article summary: {page_desc}
Why promote: {page['reason']}

Requirements:
- Concise, punchy hook
- Include the link
- No hashtags (Bluesky culture)
- Tone: conversational, insightful
- Language: English

Return ONLY the post text, nothing else."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'You write engaging social media posts for a technical blog. Be concise and punchy.'},
        {'role': 'user', 'content': bluesky_prompt}
    ],
    temperature=0.8,
    max_tokens=200,
)

bluesky_en = result['content'].strip()
print(f'=== Bluesky Post (EN) [{len(bluesky_en)} chars] ===')
print(bluesky_en)
print(f'\n{"✅ OK" if len(bluesky_en) <= 300 else "❌ TOO LONG"} ({len(bluesky_en)}/300 chars)')

=== Bluesky Post (EN) [217 chars] ===
"Your dream home might be a money pit. This podcast ep breaks down why buying could be the worst financial move you’ll ever make. Worth a listen before you sign. /blog/27/?utm_source=bluesky&utm_campaign=growth-agent"

✅ OK (217/300 chars)


## 5. Generate German Variant (Mastodon)

In [7]:
mastodon_de_prompt = f"""Schreibe einen Mastodon-Post (max 500 Zeichen) über diesen Blog-Artikel:

URL: {page_url}?utm_source=mastodon&utm_campaign=growth-agent
Titel: {page.get('title', '')}
Zusammenfassung: {page_desc}
Warum bewerben: {page['reason']}

Kontext: fretchen.eu behandelt politische Ökonomie, Spieltheorie,
Quantencomputing, Blockchain/Web3 und AI-Tools.

Anforderungen:
- Hook im ersten Satz (Frage oder starke These)
- Ein konkretes Insight aus dem Thema erwähnen
- Link einbinden
- 2-3 relevante Hashtags
- Duzen, nicht Siezen
- Ton: technisch aber verständlich, meinungsstark

Gib NUR den Post-Text zurück, nichts anderes."""

result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'Du schreibst ansprechende Social Media Posts für einen technischen Blog. Sei prägnant und pointiert.'},
        {'role': 'user', 'content': mastodon_de_prompt}
    ],
    temperature=0.8,
    max_tokens=300,
)

mastodon_de = result['content'].strip()
print(f'=== Mastodon Post (DE) [{len(mastodon_de)} chars] ===')
print(mastodon_de)
print(f'\n{"✅ OK" if len(mastodon_de) <= 500 else "❌ TOO LONG"} ({len(mastodon_de)}/500 chars)')

=== Mastodon Post (DE) [379 chars] ===
**"Mieten statt kaufen? Ein Podcast-Argument schlägt Wellen – und hat recht. 🏠💥**

Der Mythos vom Eigenheim bröckelt: Warum dein Geld in Aktien oder Crypto oft klüger arbeitet als in Beton. Spoiler: Opportunitätskosten sind der heimliche Dealbreaker.

🔗 [fretchen.eu/blog/27](https://fretchen.eu/blog/27/?utm_source=mastodon&utm_campaign=growth-agent)

#Ökonomie #Finanzen #Web3"

✅ OK (379/500 chars)


## 6. Save Drafts to Content Queue

In [8]:
from datetime import datetime
from agent.models import Draft, ContentQueue
import hashlib

def make_draft_id(content: str) -> str:
    return f'draft_{hashlib.md5(content.encode()).hexdigest()[:8]}'

drafts = [
    Draft(
        id=make_draft_id(mastodon_en),
        channel='mastodon',
        language='en',
        content=mastodon_en,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=mastodon&utm_campaign=growth-agent',
    ),
    Draft(
        id=make_draft_id(bluesky_en),
        channel='bluesky',
        language='en',
        content=bluesky_en,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=bluesky&utm_campaign=growth-agent',
    ),
    Draft(
        id=make_draft_id(mastodon_de),
        channel='mastodon',
        language='de',
        content=mastodon_de,
        source_blog_post=page.get('title', page['url']),
        link=f'{page_url}?utm_source=mastodon&utm_campaign=growth-agent',
    ),
]

queue = ContentQueue(drafts=drafts)
store.write('content_queue.json', queue)

print(f'Saved {len(drafts)} drafts to content_queue.json')
for d in drafts:
    print(f'  - {d.id} | {d.channel} ({d.language}) | {len(d.content)} chars')

Saved 3 drafts to content_queue.json
  - draft_391d4213 | mastodon (en) | 387 chars
  - draft_69be8096 | bluesky (en) | 217 chars
  - draft_8abf49f2 | mastodon (de) | 379 chars


In [9]:
llm.close()
print('Done — Content creation validated.')

Done — Content creation validated.
